In [10]:
import pandas as pd
import numpy as np
import requests
import urllib.request
import json
import geopandas as gpd
from shapely.geometry import Polygon, MultiPolygon, Point
from shapely import wkt
import seaborn as sns
import seaborn.objects as so
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import plotly.express as px
import warnings
from sklearn.neighbors import KNeighborsClassifier
import os
from scipy.stats import f_oneway
from pathlib import Path
import statsmodels.formula.api as smf

In [11]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)
pd.set_option('display.float_format', '{:.3f}'.format)
warnings.filterwarnings('ignore')

In [12]:
raw = "C:\\Users\\taavi\\Desktop\\BPHIL\\Raw data\\"
clean = "C:\\Users\\taavi\\Desktop\\BPHIL\\Clean data\\"

### Filtering logic for within Pittsburgh boundary

In [5]:
# https://pghgishub-pittsburghpa.opendata.arcgis.com/datasets/pittsburghpa::pittsburgh-boundary/explore?location=40.430398%2C-79.979853%2C11.85
geojson = gpd.read_file(raw + 'Pittsburgh_Boundary.geojson')

boundary = geojson.geometry.iloc[2]
# this is the perimeter of the city

polygons = [geom for geom in boundary.geoms]
    
def filterData(data, polygons):
    filteredData = []
    for index, row in data.iterrows():
        dataPoint = Point(row['lng'], row['lat'])
        for polygon in polygons:
            if dataPoint.within(polygon):
                filteredData.append(row)
                break
    return pd.DataFrame(filteredData)

### Neighborhoods

In [16]:
# dictionary to store temporary dfs
dfs = {}

# download the data via the provided API
offset = 0
index = 0
while True:
    url = 'https://data.wprdc.org/api/3/action/datastore_search?resource_id=668d7238-cfd2-492e-b397-51a6e74182ff&limit=32000'
    url += '&offset=' + str(offset)
    fileobj = urllib.request.urlopen(url)
    data = pd.DataFrame(json.loads(fileobj.read())['result']['records'])
    dfs['df' + str(index)] = data
    offset += 32000
    index += 1
    if len(data) == 0:
        break
        
nbrhds = pd.concat(dfs.values(), ignore_index = True)

In [61]:
nbrhds.to_csv(raw + 'raw_neighborhoods.csv', index = False)

In [62]:
nbrhds = pd.read_csv(raw + 'raw_neighborhoods.csv')

In [63]:
nbrhds = (
    nbrhds
    .assign(
        lat = nbrhds['intptlat10'].apply(lambda x: float(x[1:] if x != ' ' else np.nan)),
        lng = nbrhds['intptlon10'].apply(lambda x: float(x) if x != ' ' else np.nan)
    )
)

In [64]:
nbrhds.to_csv(clean + 'clean_nbrhds.csv', index = False)

In [5]:
nbrhds = pd.read_csv(clean + 'clean_nbrhds.csv')

### Parcels

In [40]:
# https://data.wprdc.org/dataset/parcel-centroids-in-allegheny-county-with-geographic-identifiers/resource/3fab7152-3f11-4788-8372-4c33f86ea813
# - 2010 census boundaries and redistricting prior to 2020
# dictionary to store temporary dfs
dfs = {}

# download the data via the provided API
offset = 0
index = 0
while True:
    url = 'https://data.wprdc.org/api/3/action/datastore_search?resource_id=adf1fd38-c374-4c4e-9094-5e53bd12419f&limit=32000'
    url += '&offset=' + str(offset)
    fileobj = urllib.request.urlopen(url)
    data = pd.DataFrame(json.loads(fileobj.read())['result']['records'])
    dfs['df' + str(index)] = data
    offset += 32000
    index += 1
    if len(data) == 0:
        break
        
parcels = pd.concat(dfs.values(), ignore_index = True)

In [41]:
parcels.shape

(583956, 78)

In [42]:
parcels.head(1)

,_id,OBJECTID_12,OBJECTID_1,OBJECTID,PIN,MAPBLOCKLO,MUNICODE,CALCACREAG,NOTES,PSEUDONO,COMMENTS,GlobalID,Shape_Leng,Latitude,Longitude,FID_1,OBJECTID_2,OBJECTID_3,OBJECTID_4,STATEFP10,COUNTYFP10,TRACTCE10,BLOCKCE10,GEOID10,NAME10,MTFCC10,UR10,UACE10,UATYP10,FUNCSTAT10,ALAND10,AWATER10,INTPTLAT10,INTPTLON10,geoid2,geo_name_c,geo_id_cou,level_coun,geo_name_n,geo_id_nho,level_nhoo,geo_name_t,geo_id_tra,level_trac,geo_name_b,geo_id_blo,level_bloc,geo_name_1,geo_id_c_1,level_cous,geo_name_z,geo_id_zip,level_zipc,geo_name_s,geo_id_sch,level_scho,geo_name_H,geo_id_Hou,level_Hous,geo_name_2,geo_id_Sen,level_Sena,geo_name_3,geo_id_c_2,level_co_1,OBJECTID_5,Pgh_PLI_Zo,Pgh_CityCo,Pgh_Police,Pgh_FireDi,Pgh_DPW_Di,Pgh_Ward,Distance,Shape_Le_1,Shape_Le_2,Shape_Ar_1,LatitudeStatePlane,LongitudeStatePlane
0,1,1,203,203,0627N00091000000,627-N-91,815,0.290000000000000,,,,{18151AA4-DE80-4E58-87C6-8ED7019CD783},449.145104890000027,40.540240684590437,-79.797759044177795,23,24,24,270,42,003,418000,1030,420034180001030,Block 1030,G5040,U,69697,U,S,13792.000000000000000,0.000000000000000,+40.5404591,-079.7975466,420034180001000.000000000000000,Allegheny,42003,County,,,,418000,42003418000,Census Tract,Tract 418000 Block Group 1,420034180001,Block Group,Cheswick borough,13392,County Subarea,15024,15024,Zip Code,Allegheny Valley SD,103020603,School District,PA House District 33,PAHOUSE33,PA House,PA Senate District 38,PASEN38,PA Senate,District 7,CC7,County Council,1444,0,0,0,,0,0,0.000000000000000,0.004971804650100,655.351063831000033,23906.338964599999599,446260.888842999993358,1399413.763989999890327


In [43]:
parcels['GEOID10'] = parcels['GEOID10'].astype(str)

In [44]:
parcels.to_csv(raw + 'raw_parcels.csv', index = False)

In [148]:
parcels = pd.read_csv(raw + 'raw_parcels.csv')

In [49]:
# neighborhood
parcels = parcels.loc[parcels['geo_name_n'] != ' ']
parcels['geo_name_n'] = np.where(parcels['geo_name_n'] != 'Mount Oliver Borough', parcels['geo_name_n'], 'Mt. Oliver')

# isolate 'parcelID', 'lat', 'lng', 'nbrhd', 'tract'
parcels = parcels.loc[:, ['PIN', 'Latitude', 'Longitude', 'geo_name_n', 'TRACTCE10', 'BLOCKCE10', 'GEOID10']] # could include GEOID10 for census bureau statistics mapping

# rename columns
parcels.columns = ['parcelID', 'lat', 'lng', 'nbrhd', 'tract', 'block', 'geoID'] 

# drop nulls - string 'nan' in this case
#parcels = parcels.drop(index = [583953, 583954])
exclude = [' ', 'COMMON GROUND', 'Westmoreland County', 'Washington County', 'Butler County', 'Not Assessed', 'Beaver County']
parcels = parcels.loc[parcels['parcelID'].isin(exclude) == False]

parcels = parcels.drop_duplicates(subset = 'parcelID')

# filter for just Pittsburgh
parcels = filterData(parcels, polygons)

parcels = parcels.reset_index(drop = True)

In [ ]:
parcels.shape

In [53]:
parcels.to_csv(clean + 'clean_parcels.csv', index = False)

In [26]:
parcels = pd.read_csv(clean + 'clean_parcels.csv')

### Violations - new

In [ ]:
# https://data.wprdc.org/dataset/pittsburgh-pli-violations-report/resource/70c06278-92c5-4040-ab28-17671866f81c
# dictionary to store temporary dfs
dfs = {}

# download the data via the provided API
offset = 0
index = 0
while True:
    url = 'https://data.wprdc.org/api/3/action/datastore_search?resource_id=70c06278-92c5-4040-ab28-17671866f81c&limit=32000'
    url += '&offset=' + str(offset)
    fileobj = urllib.request.urlopen(url)
    data = pd.DataFrame(json.loads(fileobj.read())['result']['records'])
    dfs['df' + str(index)] = data
    offset += 32000
    index += 1
    if len(data) == 0:
        break

viols = pd.concat(dfs.values(), ignore_index = True)

In [ ]:
viols.head(1)

In [ ]:
viols.to_csv(raw + 'raw_viols.csv', index = False)

In [4]:
viols = pd.read_csv(raw + 'raw_viols.csv')

In [ ]:
# merge in lat and lng, and also filter for within Pittsburgh
viols = viols.merge(right = parcels, left_on = 'parcel_id', right_on = 'parcelID', how = 'inner')

In [ ]:
viols.head(1)

In [ ]:
# isolate columns
viols = viols.loc[:, ['casefile_number', 'parcel_id', 'investigation_date', 'violation_code_section', 'investigation_outcome', 'lat', 'lng']]

# rename columns
viols.columns = ['violID', 'parcelID', 'date', 'code', 'outcome', 'lat', 'lng']

# drop nulls
viols = viols.dropna()

# drop duplicates
#viols = viols.drop_duplicates()

In [ ]:
viols.head(1)

In [ ]:
viols.to_csv(clean + 'clean_viols.csv', index = False)

### Violations - old

In [ ]:
# https://data.wprdc.org/dataset/pittsburgh-pli-violations-report/resource/4e5374be-1a88-47f7-afee-6a79317019b4
# dictionary to store temporary dfs
dfs = {}

# download the data via the provided API
offset = 0
index = 0
while True:
    url = 'https://data.wprdc.org/api/3/action/datastore_search?resource_id=4e5374be-1a88-47f7-afee-6a79317019b4&limit=32000'  
    url += '&offset=' + str(offset)
    fileobj = urllib.request.urlopen(url)
    data = pd.DataFrame(json.loads(fileobj.read())['result']['records'])
    dfs['df' + str(index)] = data
    offset += 32000
    index += 1
    if len(data) == 0:
        break

violsOld = pd.concat(dfs.values(), ignore_index = True)

In [ ]:
violsOld.to_csv(raw + 'raw_viols_old.csv', index = False)

In [ ]:
violsOld.head(1)

In [ ]:
violsOld = pd.read_csv(raw + 'raw_viols_old.csv')

In [ ]:
violsOld = violsOld.loc[violsOld['INSPECTION_RESULT'] == 'Violations Found']

violsOld = violsOld[['CASE_NUMBER', 'PARCEL', 'INSPECTION_DATE', 'VIOLATION', 'INSPECTION_RESULT', 'Y', 'X']]

violsOld.columns = ['violID', 'parcelID', 'date', 'code', 'outcome', 'lat', 'lng']

violsOld = violsOld.dropna()

In [ ]:
violsOld.head(1)

In [ ]:
violsOld.to_csv(clean + 'clean_viols_old.csv', index = False)

In [ ]:
violsOld = pd.read_csv(clean + 'clean_viols_old.csv')

### Crimes - 2016-2023

In [ ]:
# https://data.wprdc.org/dataset/uniform-crime-reporting-data/resource/044f2016-1dfd-4ab0-bc1e-065da05fca2e
# dictionary to store temporary dfs
dfs = {}

# download the data via the provided API
offset = 0
index = 0
while True:
    url = 'https://data.wprdc.org/api/3/action/datastore_search?resource_id=044f2016-1dfd-4ab0-bc1e-065da05fca2e&limit=32000'  
    url += '&offset=' + str(offset)
    fileobj = urllib.request.urlopen(url)
    data = pd.DataFrame(json.loads(fileobj.read())['result']['records'])
    dfs['df' + str(index)] = data
    offset += 32000
    index += 1
    if len(data) == 0:
        break

crimes1 = pd.concat(dfs.values(), ignore_index = True)

In [ ]:
crimes1.head(1)

In [ ]:
crimes1.to_csv(raw + 'raw_crimes_old.csv', index = False)

In [4]:
crimes1 = pd.read_csv(raw + 'raw_crimes_old.csv')

In [5]:
crimes1

,_id,PK,CCR,HIERARCHY,INCIDENTTIME,INCIDENTLOCATION,CLEAREDFLAG,INCIDENTNEIGHBORHOOD,INCIDENTZONE,INCIDENTHIERARCHYDESC,OFFENSES,INCIDENTTRACT,COUNCIL_DISTRICT,PUBLIC_WORKS_DIVISION,X,Y
0,1,2802309,16000001.000,10,2016-01-01T00:00:00,"400 Block North Shore DR Pittsburgh, PA 15212",Y,North Shore,1,HARRASSMENT/THREAT/ATTEMPT/PHY,2702 Aggravated Assault. / 2709(a) Harassment....,2205.000,1.000,6.000,-80.012,40.446
1,2,2803174,16004547.000,11,2016-01-01T00:01:00,"5400 Block Carnegie ST Pittsburgh, PA 15201",N,Upper Lawrenceville,2,THEFT BY DECEPTION,3922 Theft by Deception.,1011.000,7.000,2.000,-79.950,40.482
2,3,2801809,16000367.000,4,2016-01-01T00:10:00,"500 Block Mt Pleasant RD Pittsburgh, PA 15214",N,Northview Heights,1,DISCHARGE OF FIREARM INTO OCC.STRUCTURE,2707.1 Discharge of a Firearm into Occupied St...,2609.000,1.000,1.000,-80.001,40.479
3,4,2802315,16000035.000,10,2016-01-01T00:15:00,"300 Block Wood ST Pittsburgh, PA 15222",Y,Golden Triangle/Civic Arena,2,HARRASSMENT/THREAT/ATTEMPT/PHY,2709(a)(3) Harassment No Legitimate Purpose,201.000,6.000,6.000,-80.001,40.439
4,5,2802312,16000024.000,4,2016-01-01T00:16:00,"500 Block Mt Pleasant RD Pittsburgh, PA 15214",N,Northview Heights,1,PROP MISSILE INTO OCC VEHICLE/OR ROADWAY,2705 Recklessy Endangering Another Person. / 3...,2609.000,1.000,1.000,-80.001,40.479
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
340991,360742,3440790,23168219.000,99,2020-01-01T00:01:00,"1900 Block Lithgow AV Pittsburgh, PA 15214",N,Perry South,1,NaN,9129 Miscellaneous Investigation,2614.000,6.000,1.000,-80.010,40.462
340992,360772,3440804,23042039.000,99,2023-03-20T12:01:00,Zone 3,N,NaN,3,NaN,5903 Obscene and Other Sexual Materials and Pe...,NaN,NaN,NaN,NaN,NaN
340993,360773,3440881,23157117.000,99,2023-10-03T10:16:00,"10 Block E Sycamore ST Pittsburgh, PA 15211",Y,Mount Washington,3,NaN,9490 Missing Persons (18 and Over),1914.000,2.000,5.000,-80.007,40.431
340994,360774,3440882,23167843.000,99,2023-10-22T11:45:00,"300 Block N Craig ST Pittsburgh, PA 15213",Y,North Oakland,4,NaN,9489 Found Property,507.000,8.000,3.000,-79.949,40.446


In [7]:
part1_offenses = [
    'HOMICIDE', 'RAPE', 'ROBBERY', 'AGGRAVATED ASSAULT', 'BURGLARY', 'ARSON', 'THEFT', 'TRAFFICKING', 'ASSAULT', 'MURDER'
]

part2_offenses = [
    'OTHER ASSAULTS', 'FORGERY AND COUNTERFEITING', 'FRAUD', 'EMBEZZLEMENT', 'STOLEN PROPERTY', 'VANDALISM', 'WEAPONS', 'FIREARM', 'PROSTITUTION',
    'SEX OFFENSES','DRUG ABUSE VIOLATIONS', 'GAMBLING', 'OFFENSES AGAINST THE FAMILY AND CHILDREN', 'DRIVING UNDER THE INFLUENCE', 'LIQUOR',
    'DRUNKENNESS', 'DISORDERLY CONDUCT', 'VAGRANCY', 'ALL OTHER OFFENSES', 'SUSPICION', 'CURFEW', 'LOITERING',
]

In [8]:
def classify_ucr(desc):
    if pd.isna(desc):
        return 'Unknown'
    desc_upper = desc.upper()
    for offense in part1_offenses:
        if offense in desc_upper:
            return 'High'
    for offense in part2_offenses:
        if offense in desc_upper:
            return 'Low'
    return 'Low'

In [9]:
crimes1['severity'] = crimes1['OFFENSES'].apply(classify_ucr)

In [10]:
crimes1 = crimes1[['OFFENSES', 'severity', 'INCIDENTTIME', 'INCIDENTLOCATION', 'INCIDENTNEIGHBORHOOD', 'INCIDENTZONE', 'INCIDENTTRACT', 'Y', 'X']]

crimes1.columns = ['crime', 'severity', 'date', 'address', 'nbrhd', 'zone', 'tract', 'lat', 'lng']

crimes1 = filterData(crimes1, polygons).reset_index(drop = True)

# crimes1 = crimes1.drop_duplicates(subset = ['crime', 'date', 'address'])
# not dropping duplicates because data disclaimer says location is block-address rather than street-address to protect identity
# so I can't say that dupe addresses or even dupe coordinates are also dupe incidents
# https://www.pittsburghpa.gov/files/assets/city/v/1/public-safety/documents/police/2025_pittsburgh_monthly_incident_dataset_disclaimer.pdf

# crimes1['date'] = pd.to_datetime(crimes1['date'])

In [11]:
(crimes1[['lat', 'lng']] == 0).sum()

lat    0
lng    0
dtype: int64

In [12]:
crimes1.to_csv(clean + 'clean_crimes_old.csv', index = False)

In [ ]:
crimes1 = pd.read_csv(clean + 'clean_crimes_old.csv')

### Crimes - 2024-2025

In [ ]:
# https://data.wprdc.org/dataset/monthly-criminal-activity-dashboard/resource/bd41992a-987a-4cca-8798-fbe1cd946b07
# dictionary to store temporary dfs
dfs = {}

# download the data via the provided API
offset = 0
index = 0
while True:
    url = 'https://data.wprdc.org/api/3/action/datastore_search?resource_id=bd41992a-987a-4cca-8798-fbe1cd946b07&limit=32000'  
    url += '&offset=' + str(offset)
    fileobj = urllib.request.urlopen(url)
    data = pd.DataFrame(json.loads(fileobj.read())['result']['records'])
    dfs['df' + str(index)] = data
    offset += 32000
    index += 1
    if len(data) == 0:
        break

crimes2 = pd.concat(dfs.values(), ignore_index = True)

In [ ]:
crimes2.to_csv(raw + 'raw_crimes_new.csv', index = False)

In [13]:
crimes2 = pd.read_csv(raw + 'raw_crimes_new.csv')

In [14]:
crimes2['severity'] = crimes2['Violation'].apply(classify_ucr)

crimes2 = (
    crimes2
    .assign(
        sev_code = np.where(crimes2['severity'] == 'High', 1, 0)
    )
    .assign(
        sev_code = lambda x: x.groupby('Report_Number')['sev_code'].transform('max')
    )
    .assign(
        severity = lambda x: np.where(x['sev_code'] == 1, 'High', 'Low')
    )
    .drop_duplicates(subset = 'Report_Number')
)

In [15]:
crimes2 = crimes2[['Violation', 'severity', 'ReportedDate', 'Block_Address', 'Neighborhood', 'Zone', 'Tract', 'YCOORD', 'XCOORD']]

crimes2.columns = ['crime', 'severity', 'date', 'address', 'nbrhd', 'zone', 'tract', 'lat', 'lng']

crimes2 = filterData(crimes2, polygons).reset_index(drop = True)

# https://www.pittsburghpa.gov/files/assets/city/v/1/public-safety/documents/police/2025_pittsburgh_monthly_incident_dataset_disclaimer.pdf

In [16]:
crimes2.to_csv(clean + 'clean_crimes_new.csv', index = False)

In [ ]:
crimes2 = pd.read_csv(clean + 'clean_crimes_new.csv')

### Concatenating crimes

In [17]:
crimes = pd.concat([crimes1, crimes2], axis = 0).reset_index(drop = True)

In [18]:
crimes = crimes.loc[pd.to_datetime(crimes['date'], format = 'mixed').dt.year >= 2020]

In [19]:
crimes.to_csv(clean + 'clean_crimes.csv', index = False)

In [7]:
crimes = pd.read_csv(clean + 'clean_crimes.csv')

### Pulling sociodemographic data (Race, Poverty, Age, Employment, Education, Income, Sex)
Typical call structure:
- https://api.census.gov/data/YEAR/DATASET
  - ?get=VARIABLES
  - &for=GEOGRAPHY
  - &in=GEOGRAPHY_FILTER
  - &key=API_KEY


In [41]:
api_key = 'a2a5d7f9f5a69b9eeb5aac07162cd7b65680ae20'
state = '42' # Pennslvania
county = '003' # Allegheny County
dataset = '2022/acs/acs5'
geography = 'block group:*'

In [57]:
sex_vars = {
    'total_pop': 'B01001_001E',
    'male': 'B01001_002E',
    'female': 'B01001_026E'
}

race_vars = {
    'total_pop': 'B02001_001E',
    'white': 'B02001_002E',
    'black': 'B02001_003E',
    'native': 'B02001_004E',
    'asian': 'B02001_005E',
    'islander': 'B02001_006E',
    'other': 'B02001_007E',
    'mixed': 'B02001_008E'
}

empl_vars = {
    'total_pop_16+': 'B23025_001E', # age 16+
    'labor_force': 'B23025_002E',
    'employed': 'B23025_004E',
    'unemployed': 'B23025_005E'
}

poverty_vars = {
    'total_pop_with_status': 'B17001_001E', # total population for whom poverty status is determined at all
    'impoverished': 'B17001_002E'
}

income_vars = {
    'median_household_income': 'B19013_001E'
}

edu_vars = {
    'total_pop_25+': 'B15003_001E', # age 25+
    'high_school': 'B15003_017E',
    'bachelors': 'B15003_021E',
    'masters': 'B15003_022E',
    'professional': 'B15003_023E',
    'doctorate': 'B15003_024E'
}

age_sex_vars = {
    "total_pop": "B01001_001E",
    "male_-5": "B01001_003E","male_5_9": "B01001_004E","male_10_14": "B01001_005E",
    "male_15_17": "B01001_006E","male_18_19": "B01001_007E","male_20": "B01001_008E","male_21": "B01001_009E","male_22_24": "B01001_010E",
    "male_25_29": "B01001_011E","male_30_34": "B01001_012E","male_35_39": "B01001_013E","male_40_44": "B01001_014E","male_45_49": "B01001_015E",
    "male_50_54": "B01001_016E","male_55_59": "B01001_017E","male_60_61": "B01001_018E","male_62_64": "B01001_019E","male_65_66": "B01001_020E",
    "male_67_69": "B01001_021E","male_70_74": "B01001_022E","male_75_79": "B01001_023E","male_80_84": "B01001_024E","male_85+": "B01001_025E",
    "female_-5": "B01001_027E","female_5_9": "B01001_028E","female_10_14": "B01001_029E",
    "female_15_17": "B01001_030E","female_18_19": "B01001_031E","female_20": "B01001_032E","female_21": "B01001_033E","female_22_24": "B01001_034E",
    "female_25_29": "B01001_035E","female_30_34": "B01001_036E","female_35_39": "B01001_037E","female_40_44": "B01001_038E","female_45_49": "B01001_039E",
    "female_50_54": "B01001_040E","female_55_59": "B01001_041E","female_60_61": "B01001_042E","female_62_64": "B01001_043E","female_65_66": "B01001_044E",
    "female_67_69": "B01001_045E","female_70_74": "B01001_046E","female_75_79": "B01001_047E","female_80_84": "B01001_048E","female_85+": "B01001_049E",
}


In [43]:
all_vars = {
    'sex': sex_vars,
    'race': race_vars,
    'employment': empl_vars,
    'poverty': poverty_vars,
    'income': income_vars,
    'education': edu_vars,
    'age': age_sex_vars
}

In [44]:
url = f"https://api.census.gov/data/{dataset}"
params = {
    "get": "NAME",
    "for": "tract:*",
    "in": f"state:{state}+county:{county}",
    "key": api_key
}

r = requests.get(url, params=params)
r.raise_for_status()
data = r.json()

# Extract tract codes
tract_codes = [row[3] for row in data[1:]]

In [13]:
parcels = pd.read_csv(clean + 'clean_parcels.csv')

In [46]:
parcels['tract'] = parcels['tract'].astype(str)
no_dupes = parcels.drop_duplicates(subset = 'tract')[['tract']]
no_dupes['clean_tract'] = np.where(no_dupes['tract'].str.len() < 6, '0' + no_dupes['tract'], no_dupes['tract'])
parcels = parcels.merge(right = no_dupes, on = 'tract', how = 'left')
parcels['tract'] = parcels['clean_tract']
parcels = parcels.drop(columns = 'clean_tract')

In [47]:
pittsburgh_tracts = parcels['tract'].astype(str).unique()
tract_codes = [tract for tract in tract_codes if tract in pittsburgh_tracts]

In [48]:
len(tract_codes), parcels['tract'].nunique(), round(len(tract_codes) / parcels['tract'].nunique(), 3)

(112, 138, 0.812)

Note: we're only keeping 81.2% of the tracts in all of Pittsburgh

In [50]:
base = f'https://api.census.gov/data/{dataset}'

def pull_data(var_dict, geography, geo_filter, api_key):
    vars_list = list(var_dict.values())
    vars_names = list(var_dict.keys())
    get_param = 'NAME,' + ','.join(vars_list)

    params = {
        'get': get_param,
        'for': geography,
        'in': geo_filter,
        'key': api_key
    }

    r = requests.get(base, params = params)
    r.raise_for_status()

    data = r.json()
    df = pd.DataFrame(data[1:], columns = data[0])

    df = df.iloc[:, list(range(1, df.shape[1] - 4)) + [-2] + [-1]]

    for col in vars_list:
        df[col] = pd.to_numeric(df[col], errors = 'coerce')

    df.columns = vars_names + ['tract', 'block_group']

    return df

In [260]:
for name, vars in zip(all_vars.keys(), all_vars.values()):

    full = pd.DataFrame()

    for tract in tract_codes:
        geography = 'block group:*'
        geo_filter = f"state:{state}+county:{county}+tract:{tract}"

        sub = pull_data(vars, geography, geo_filter, api_key)

        full = pd.concat([full, sub], axis = 0)

    full.to_csv(raw + f'{name}_ACS_data.csv', index = False)

Poverty specifically, bc this requires census tract level

In [ ]:
def pull_data(var_dict, geography, geo_filter, api_key):
    vars_list = list(var_dict.values())
    vars_names = list(var_dict.keys())

    params = {
        'get': 'NAME,' + ','.join(vars_list),
        'for': geography,
        'in': geo_filter,
        'key': api_key
    }

    r = requests.get(base, params=params)
    r.raise_for_status()

    data = r.json()
    df = pd.DataFrame(data[1:], columns=data[0])

    # Keep only variables + geography columns
    geo_cols = [c for c in df.columns if c not in vars_list and c != 'NAME']
    df = df[vars_list + geo_cols]

    for col in vars_list:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    # Rename variables
    df = df.rename(columns=dict(zip(vars_list, vars_names)))

    return df

In [60]:
for tract in tract_codes:
    geography = "tract:*"
    geo_filter = f"state:{state}+county:{county}"

    sub = pull_data(poverty_vars, geography, geo_filter, api_key)

In [ ]:
sub['impoverished'] = sub['impoverished'] / sub['total_pop_with_status']
sub = sub.fillna(0)

In [67]:
sub.to_csv(clean + 'poverty_ACS_data.csv', index = False)

#### Cleaning and making everything proportions

In [318]:
dfs = {}
for key in all_vars.keys():
    dfs[key] = pd.read_csv(raw + f'{key}_ACS_data.csv')

Sex

In [306]:
sex = dfs['sex']
for col in sex.iloc[:, 1:-2].columns:
    sex[col] = sex[col] / sex['total_pop']
sex = sex.fillna(0)
sex.to_csv(clean + 'sex_ACS_data.csv', index = False)

Race

In [307]:
race = dfs['race']
for col in race.iloc[:, 1:-2].columns:
    race[col] = race[col] / race['total_pop']
race = race.fillna(0)
race.to_csv(clean + 'race_ACS_data.csv', index = False)

Employment

In [319]:
empl = dfs['employment']
for col in empl.iloc[:, 1:-2].columns:
    empl[col] = empl[col] / empl['total_pop_16+']
empl = empl.fillna(0)
empl.to_csv(clean + 'employment_ACS_data.csv', index = False)

Income

In [315]:
income = dfs['income']
income = income.loc[income['median_household_income'] != -666666666]
income.to_csv(clean + 'income_ACS_data.csv', index = False)

Education

In [321]:
edu = dfs['education']
for col in edu.iloc[:, 1:-2].columns:
    edu[col] = edu[col] / edu['total_pop_25+']
edu = edu.fillna(0)
edu.to_csv(clean + 'education_ACS_data.csv', index = False)

Age

In [323]:
df_age = dfs['age']

age_bins = range(1, 24)

for i in age_bins:
    col = df_age.iloc[:, i:-2:23]
    sums = col.sum(axis = 1)
    name = col.columns[0][5:]
    df_age[name] = sums

    df_age[name] = df_age[name] / df_age['total_pop']

df_age = df_age.iloc[:, [47, 48, 0] + [i for i in range(49, df_age.shape[1])]].fillna(0)

df_age.to_csv(clean + 'age_ACS_data.csv', index = False)

### Pulling QoL data
I was looking into the US Census Bureau's Survey of Income and Program Participation (SIPP) as a measure of 'well-being', but the data isn't available at any level more granular than the city. I can't use that, because my whole objectie is to be more granular than that.

### Street shapefiles

In [ ]:
# dictionary to store temporary dfs
dfs = {}

# download the data via the provided API
offset = 0
index = 0
while True:
    url = 'https://data.wprdc.org/api/3/action/datastore_search?resource_id=453e8677-66b0-45c0-8083-b8955ac742c7&limit=32000'
    url += '&offset=' + str(offset)
    fileobj = urllib.request.urlopen(url)
    data = pd.DataFrame(json.loads(fileobj.read())['result']['records'])
    dfs['df' + str(index)] = data
    offset += 32000
    index += 1
    if len(data) == 0:
        break

shapefiles = pd.concat(dfs.values(), ignore_index = True)

In [ ]:
shapefiles.to_csv(raw + 'raw_shapefiles.csv', index = False)

In [ ]:
shapefiles.head(1)

In [ ]:
names = []
types = []
lngs = []
lats = []
for idx, street in shapefiles[['st_name', 'st_type', 'lmuni', 'rmuni']].value_counts().reset_index().iterrows():
    values = shapefiles.loc[
    (shapefiles['st_name'] == street['st_name']) &
    (shapefiles['st_type'] == street['st_type']) &
    (shapefiles['lmuni'] == street['lmuni']) &
    (shapefiles['rmuni'] == street['rmuni']),
    'wkt'].values

    for i in range(len(values)):
        lngs.extend([float(j.strip().split(' ')[0]) for j in values[i][12:-1].split(',')])
        lats.extend([float(j.strip().split(' ')[1]) for j in values[i][12:-1].split(',')])
        n = len([float(j.strip().split(' ')[0]) for j in values[i][12:-1].split(',')])
        names.extend([street['st_name']] * n)
        types.extend([street['st_type']] * n)

In [ ]:
len(names), len(types), len(lats), len(lngs)

In [ ]:
streets = pd.DataFrame({'name': names, 'type': types, 'lat': lats, 'lng': lngs})

In [ ]:
streets = filterData(streets, polygons)

In [ ]:
streets.to_csv(clean + 'street_shapefiles.csv', index = False)

In [ ]:
#streets = pd.read_csv(clean + 'street_shapefiles.csv')

In [ ]:
streets

In [ ]:
# fig = px.scatter_mapbox(streets.iloc[:500], lat = 'lat', lon = 'lng', zoom = 10)
# fig.update_layout(mapbox_style = 'open-street-map')
# fig.show()

### Permits (for demolitions)

In [ ]:
# https://data.wprdc.org/dataset/pli-permits/resource/f4d1177a-f597-4c32-8cbf-7885f56253f6
# dictionary to store temporary dfs
dfs = {}

# download the data via the provided API
offset = 0
index = 0
while True:
    url = 'https://data.wprdc.org/api/3/action/datastore_search?resource_id=f4d1177a-f597-4c32-8cbf-7885f56253f6&limit=32000'
    url += '&offset=' + str(offset)
    fileobj = urllib.request.urlopen(url)
    data = pd.DataFrame(json.loads(fileobj.read())['result']['records'])
    dfs['df' + str(index)] = data
    offset += 32000
    index += 1
    if len(data) == 0:
        break

permits = pd.concat(dfs.values(), ignore_index = True)

In [ ]:
permits.to_csv(raw + 'raw_permits.csv', index = False)

In [5]:
permits = pd.read_csv(raw + 'raw_permits.csv')

In [15]:
development = (
    permits
    .query('permit_type.isin(["BUILDING", "Building & Development Application"]) & status == "Completed" & @pd.to_datetime(issue_date).dt.year >= 2020')
    [['parcel_num', 'issue_date', 'permit_id', 'permit_type', 'owner_name', 'work_type', 'work_description', 'commercial_or_residential',
                   'total_project_value', 'latitude', 'longitude', 'neighborhood', 'status']]
)

development.columns = ['parcelID', 'date', 'permitID', 'permitType', 'ownerName', 'workType', 'desc', 'comm_res', 'value', 'lat', 'lng', 'neighborhood', 'status']

development['date'] = pd.to_datetime(development['date'], format = 'mixed')

development = development.loc[(development['date'].dt.year >= 2020)]

In [16]:
development.to_csv(clean + 'clean_development_expenditures.csv', index = False)

In [ ]:
permits.head(1)

,_id,permit_id,permit_type,owner_name,contractor_name,work_description,work_type,commercial_or_residential,total_project_value,issue_date,parcel_num,address,latitude,longitude,council_district,neighborhood,ward,zip_code,status
0,3548939,DP-2024-10311,Demolition Permit,EAA HOLDINGS LLC,Lutterman Excavation LLC,DEMO 2 STORY SINGLE FAMILY DETACHED DWELLING,COMPLETE DEMOLITION,Residential,16200.000,2024-09-05,0060L00317000000,No primary address specified,40.398,-79.988,4.000,Carrick,29.000,15210,Completed


In [ ]:
permits = permits[['parcel_num', 'issue_date', 'permit_id', 'permit_type', 'owner_name', 'work_type', 'work_description', 'commercial_or_residential',
                   'total_project_value', 'latitude', 'longitude', 'neighborhood', 'status']]

permits.columns = ['parcelID', 'date', 'permitID', 'permitType', 'ownerName', 'workType', 'desc', 'comm_res', 'value', 'lat', 'lng', 'neighborhood', 'status']

permits['date'] = pd.to_datetime(permits['date'], format = 'mixed')

permits = permits.loc[(permits['date'].dt.year >= 2020) & (permits['permitType'] == 'Demolition Permit')]

In [ ]:
permits.to_csv(clean + 'clean_demos_new.csv', index = False)

In [15]:
demos_new = pd.read_csv(clean + 'clean_demos_new.csv')

In [ ]:
#demos_new.to_csv(clean + 'clean_demos.csv', index = False)

### Historical demolitions preceding 2020 (local CSVs manually downloaded from WPRDC)

Only going back to 2016 because that is as far back as the robust crimes reporting goes, and it is as far back as the code violations goes

In [ ]:
# https://data.wprdc.org/dataset/city-of-pittsburgh-building-permit-summary

folder = Path(raw + 'Historical permits data')

excels = folder.glob('*.xls*')
dfs = []

for i, file in enumerate(excels, start = 1):
    try:
        dfs.append(pd.read_excel(file))
    except:
        print(i, file.name)

In [ ]:
# dict to hold combined dfs per unique column set
combined = {}

for i, df in enumerate(dfs, start=1):
    cols = tuple(df.columns)  # or frozenset(df.columns) if order doesn’t matter
    
    if cols not in combined:
        # start a new combined df for this column set
        combined[cols] = df.copy()
        #print(f"Started new group at DataFrame {i}")
    else:
        # append to the existing combined df
        combined[cols] = pd.concat([combined[cols], df], ignore_index=True)
        #print(f"Appended DataFrame {i} to existing group")

combined = {i: df for i, df in enumerate(combined.values())}

In [ ]:
cols_renames = [
    ['permitID', 'parcelID', 'address', 'ward', 'owner', 'contractor', 'type', 'structure', 'desc', 'value', 'year'],
    ['permitID', 'year', 'parcelID', 'address', 'neighborhood', 'ward', 'owner', 'contractor', 'type', 'structure', 'desc', 'value'],
    ['permitID', 'ward', 'parcelID', 'address', 'owner', 'contractor', 'desc', 'value', 'year'],
    ['permitID', 'year', 'parcelID', 'address', 'neighborhood', 'ward', 'owner', 'contractor', 'type', 'structure', 'desc', 'value'],
    ['permitID', 'year', 'parcelID', 'address', 'neighborhood', 'ward', 'owner', 'contractor', 'type', 'structure', 'desc', 'value'],
    ['year', 'permitID', 'ward', 'parcelID', 'address', 'owner', 'contractor', 'desc', 'value'],
    ['permitID', 'ward', 'parcelID', 'address', 'owner', 'contractor', 'desc', 'value', 'year'],
    ['permitID', 'ward', 'parcelID', 'address', 'owner', 'contractor', 'desc', 'value', 'year']
]

cols_keep = ['permitID', 'parcelID', 'year']
# not all the data has 'type' variable to determine if the demo is partial or complete - will just use all demos including non-complete ones

for i, names in zip(combined, cols_renames):
    combined[i].columns = names
    combined[i] = combined[i].dropna().loc[lambda x: x['permitID'].str.contains('D')]#, cols_keep]

# demos = pd.concat([combined[i] for i in combined], axis = 0).reset_index(drop = True)
# demos['parcelID'] = demos['parcelID'].str.replace('-', '')
# demos = demos.loc[demos['parcelID'].str.len() == 16]
# demos['year'] = pd.to_datetime(demos['year'])
# demos.loc[demos['year'].astype(str).str[-4:] != '0000', 'year'] = demos['year'].astype(str).str[-4:]
# demos['year'] = demos['year'].dt.year

In [ ]:
demos_new['status'].value_counts(normalize = True)

status
Completed                          0.755
Issued                             0.121
Expired                            0.114
Revoked                            0.003
Application Finalization           0.003
Amendment Application Incomplete   0.002
Amendment Applicant Revisions      0.001
Name: proportion, dtype: float64

In [ ]:
combined[0]['desc'].value_counts(normalize = True)

desc
RAZE GARAGE                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            0.049
DEMOLISH STRUCTURE, CAP UTILITIES, BACKFILL AND GRADE                                                                                                                                                                                                                                                                                                                 

In [ ]:
demos.to_csv(clean + 'clean_demos_old.csv', index = False)

In [ ]:
demos_old = pd.read_csv(clean + 'clean_demos_old.csv')

In [ ]:
demos_new['year'] = pd.to_datetime(demos_new['date'], format = 'mixed').dt.year
demos_new = demos_new[['permitID', 'parcelID', 'year']]

In [ ]:
demos = pd.concat([demos_new, demos_old], axis = 0).sort_values(by = 'year').reset_index(drop = True)

In [ ]:
demos = demos.merge(right = parcels[['parcelID', 'lat', 'lng', 'nbrhd']], on = 'parcelID', how = 'inner')

In [ ]:
demos.to_csv(clean + 'clean_demos.csv', index = False)

In [ ]:
demos = pd.read_csv(clean + 'clean_demos.csv')

### Condemnations

In [47]:
# https://data.wprdc.org/dataset/condemned-properties/resource/0a963f26-eb4b-4325-bbbc-3ddf6a871410
# dictionary to store temporary dfs
dfs = {}

# download the data via the provided API
offset = 0
index = 0
while True:
    url = 'https://data.wprdc.org/api/action/datastore_search?resource_id=0a963f26-eb4b-4325-bbbc-3ddf6a871410&limit=32000'
    url += '&offset=' + str(offset)
    fileobj = urllib.request.urlopen(url)
    data = pd.DataFrame(json.loads(fileobj.read())['result']['records'])
    dfs['df' + str(index)] = data
    offset += 32000
    index += 1
    if len(data) == 0:
        break

conds = pd.concat(dfs.values(), ignore_index = True)

In [48]:
conds.to_csv(raw + 'raw_condemnations.csv', index = False)

In [32]:
conds = pd.read_csv(raw + 'raw_condemnations.csv')

In [33]:
conds = conds.rename(columns = {'parcel_id': 'parcelID', 'property_type': 'status'})

conds = conds.loc[conds['status'] == 'Condemned Property'].reset_index(drop = True)

conds['year'] = pd.to_datetime(conds['date']).dt.year

conds = conds.loc[conds['year'] >= 2020].reset_index(drop = True)

conds = conds[['parcelID', 'year']]

In [35]:
conds.to_csv(clean + 'clean_condemnations.csv', index = False)

In [ ]:
conds = pd.read_csv(clean + 'clean_condemnations.csv')

### Vacancies

In [325]:
# https://data.wprdc.org/dataset/vacant-addresses/resource/70dd02d2-137d-43c9-b158-f7b1ec6c6d42
# dictionary to store temporary dfs
dfs = {}

# download the data via the provided API
offset = 0
index = 0
while True:
    url = 'https://data.wprdc.org/api/action/datastore_search?resource_id=0d97c478-cce1-488d-aabc-da116cc6987d&limit=32000'
    url += '&offset=' + str(offset)
    fileobj = urllib.request.urlopen(url)
    data = pd.DataFrame(json.loads(fileobj.read())['result']['records'])
    dfs['df' + str(index)] = data
    offset += 32000
    index += 1
    if len(data) == 0:
        break

vacs = pd.concat(dfs.values(), ignore_index = True)

In [326]:
vacs.to_csv(raw + 'raw_vacancies.csv', index = False)

In [11]:
vacs = pd.read_csv(raw + 'raw_vacancies.csv')

In [13]:
vacs = vacs[['geoid', 'quarter_sortable', 'total_address_count', 'total_vac_count']]
vacs.columns = ['geoID', 'quarter', 'total_addresses', 'total_vacancies']
vacs['year'] = vacs['quarter'].str[0:4].astype(int)
vacs = vacs.query('year >= 2020')
vacs['prop_vacant'] = vacs['total_vacancies'] / vacs['total_addresses']
vacs = vacs.reset_index(drop = True)
vacs['tract'] = vacs['geoID'].astype(str).str[-6:]

In [14]:
vacs.to_csv(clean + 'clean_vacancies.csv', index = False)

In [6]:
vacs = pd.read_csv(clean + 'clean_vacancies.csv')

In [23]:
vacs.loc[vacs['tract'].astype(str).isin(parcels['tract'].unique())]

,geoID,quarter,total_addresses,total_vacancies,year,prop_vacant,tract


I'm having trouble connecting parcels and vacancies - I'm not sure if there is any meaningful overlap - I probably can't do analysis with this.

### Property Sales

In [ ]:
# https://data.wprdc.org/dataset/real-estate-sales/resource/5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1
# dictionary to store temporary dfs
dfs = {}

# download the data via the provided API
offset = 0
index = 0
while True:
    url = 'https://data.wprdc.org/api/3/action/datastore_search?resource_id=5bbe6c55-bce6-4edb-9d04-68edeb6bf7b1&limit=32000'
    url += '&offset=' + str(offset)
    fileobj = urllib.request.urlopen(url)
    data = pd.DataFrame(json.loads(fileobj.read())['result']['records'])
    dfs['df' + str(index)] = data
    offset += 32000
    index += 1
    if len(data) == 0:
        break

sales = pd.concat(dfs.values(), ignore_index = True)

In [ ]:
sales.shape

In [ ]:
sales.head(1)

In [ ]:
sales.to_csv(raw + 'raw_sales.csv', index = False)

In [7]:
sales = pd.read_csv(raw + 'raw_sales.csv')

In [8]:
sales = sales[['PARID', 'SALEDATE', 'PRICE']]

sales.columns = ['parcelID', 'date', 'value']

### Appeals 1

In [ ]:
# https://data.wprdc.org/api/3/action/datastore_search?resource_id=8eff881d-4d28-4064-83f1-30cc991cfec7
# dictionary to store temporary dfs
dfs = {}

# download the data via the provided API
offset = 0
index = 0
while True:
    url = 'https://data.wprdc.org/api/3/action/datastore_search?resource_id=65855e14-549e-4992-b5be-d629afc676fa&limit=32000'
    url += '&offset=' + str(offset)
    fileobj = urllib.request.urlopen(url)
    data = pd.DataFrame(json.loads(fileobj.read())['result']['records'])
    dfs['df' + str(index)] = data
    offset += 32000
    index += 1
    if len(data) == 0:
        break

appeals1 = pd.concat(dfs.values(), ignore_index = True)

In [ ]:
appeals1.shape

In [ ]:
appeals1.head(1)

,_id,PARID,PROPERTYHOUSENUM,PROPERTYFRACTION,PROPERTYADDRESS,PROPERTYCITY,PROPERTYSTATE,PROPERTYUNIT,PROPERTYZIP,MUNICODE,MUNIDESC,SCHOOLCODE,SCHOOLDESC,LEGAL1,LEGAL2,LEGAL3,NEIGHCODE,NEIGHDESC,TAXCODE,TAXDESC,TAXSUBCODE,TAXSUBCODE_DESC,OWNERCODE,OWNERDESC,CLASS,CLASSDESC,USECODE,USEDESC,LOTAREA,HOMESTEADFLAG,CLEANGREEN,FARMSTEADFLAG,ABATEMENTFLAG,RECORDDATE,SALEDATE,SALEPRICE,SALECODE,SALEDESC,DEEDBOOK,DEEDPAGE,PREVSALEDATE,PREVSALEPRICE,PREVSALEDATE2,PREVSALEPRICE2,CHANGENOTICEADDRESS1,CHANGENOTICEADDRESS2,CHANGENOTICEADDRESS3,CHANGENOTICEADDRESS4,COUNTYBUILDING,COUNTYLAND,COUNTYTOTAL,COUNTYEXEMPTBLDG,LOCALBUILDING,LOCALLAND,LOCALTOTAL,FAIRMARKETBUILDING,FAIRMARKETLAND,FAIRMARKETTOTAL,STYLE,STYLEDESC,STORIES,YEARBLT,EXTERIORFINISH,EXTFINISH_DESC,ROOF,ROOFDESC,BASEMENT,BASEMENTDESC,GRADE,GRADEDESC,CONDITION,CONDITIONDESC,CDU,CDUDESC,TOTALROOMS,BEDROOMS,FULLBATHS,HALFBATHS,HEATINGCOOLING,HEATINGCOOLINGDESC,FIREPLACES,BSMTGARAGE,FINISHEDLIVINGAREA,CARDNUMBER,ALT_ID,TAXYEAR,ASOFDATE
0,14151259,0001G00106000000,0.000,,MARKET ST,PITTSBURGH,PA,,15222.000,101,1st Ward - PITTSBURGH,47,Pittsburgh,LOT 38.67X60.19,PT EXTINGUISHED ALLEY = 21.75X4X21.75X4,NaN,51C02,PITTSBURGH URBAN,T,20 - Taxable,NaN,NaN,20,CORPORATION,C,COMMERCIAL,400,VACANT COMMERCIAL LAND,2415,NaN,NaN,NaN,NaN,12-30-2016,11-11-2016,10.000,H,MULTI-PARCEL SA,16656,25.000,12-18-1991,210000.000,NaN,NaN,2020 SMALLMAN ST STE 301,,PITTSBURGH PA,15222.000,0,61400,61400,0,0,61400,61400,0,61400,61400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025,2025-08-01


In [ ]:
appeals1.to_csv(raw + 'appeals1.csv', index = False)

In [5]:
appeals1 = pd.read_csv(raw + 'appeals1.csv')

In [6]:
appeals1

,_id,PARID,PROPERTYHOUSENUM,PROPERTYFRACTION,PROPERTYADDRESS,PROPERTYCITY,PROPERTYSTATE,PROPERTYUNIT,PROPERTYZIP,MUNICODE,MUNIDESC,SCHOOLCODE,SCHOOLDESC,LEGAL1,LEGAL2,LEGAL3,NEIGHCODE,NEIGHDESC,TAXCODE,TAXDESC,TAXSUBCODE,TAXSUBCODE_DESC,OWNERCODE,OWNERDESC,CLASS,CLASSDESC,USECODE,USEDESC,LOTAREA,HOMESTEADFLAG,CLEANGREEN,FARMSTEADFLAG,ABATEMENTFLAG,RECORDDATE,SALEDATE,SALEPRICE,SALECODE,SALEDESC,DEEDBOOK,DEEDPAGE,PREVSALEDATE,PREVSALEPRICE,PREVSALEDATE2,PREVSALEPRICE2,CHANGENOTICEADDRESS1,CHANGENOTICEADDRESS2,CHANGENOTICEADDRESS3,CHANGENOTICEADDRESS4,COUNTYBUILDING,COUNTYLAND,COUNTYTOTAL,COUNTYEXEMPTBLDG,LOCALBUILDING,LOCALLAND,LOCALTOTAL,FAIRMARKETBUILDING,FAIRMARKETLAND,FAIRMARKETTOTAL,STYLE,STYLEDESC,STORIES,YEARBLT,EXTERIORFINISH,EXTFINISH_DESC,ROOF,ROOFDESC,BASEMENT,BASEMENTDESC,GRADE,GRADEDESC,CONDITION,CONDITIONDESC,CDU,CDUDESC,TOTALROOMS,BEDROOMS,FULLBATHS,HALFBATHS,HEATINGCOOLING,HEATINGCOOLINGDESC,FIREPLACES,BSMTGARAGE,FINISHEDLIVINGAREA,CARDNUMBER,ALT_ID,TAXYEAR,ASOFDATE
0,14151259,0001G00106000000,0.000,,MARKET ST,PITTSBURGH,PA,,15222.000,101,1st Ward - PITTSBURGH,47,Pittsburgh,LOT 38.67X60.19,PT EXTINGUISHED ALLEY = 21.75X4X21.75X4,NaN,51C02,PITTSBURGH URBAN,T,20 - Taxable,NaN,NaN,20,CORPORATION,C,COMMERCIAL,400,VACANT COMMERCIAL LAND,2415,NaN,NaN,NaN,NaN,12-30-2016,11-11-2016,10.000,H,MULTI-PARCEL SA,16656,25.000,12-18-1991,210000.000,NaN,NaN,2020 SMALLMAN ST STE 301,,PITTSBURGH PA,15222.000,0,61400,61400,0,0,61400,61400,0,61400,61400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025,2025-08-01
1,14151260,0001G00107000000,100.000,,MARKET ST,PITTSBURGH,PA,,15222.000,101,1st Ward - PITTSBURGH,47,Pittsburgh,WOODS PLAN PT 226 LOT 20X60.19,PT EXTINGUISHED ALLEY = 20X4X20X4,NaN,51C02,PITTSBURGH URBAN,T,20 - Taxable,NaN,NaN,20,CORPORATION,C,COMMERCIAL,400,VACANT COMMERCIAL LAND,1284,NaN,NaN,NaN,NaN,09-03-2004,09-03-2004,344250.000,H,MULTI-PARCEL SA,12182,537.000,04-21-2004,67987.000,04-14-1997,1.000,104 MARKET ST,,PITTSBURGH PA,15222.000,0,51400,51400,0,0,51400,51400,0,51400,51400,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025,2025-08-01
2,14151261,0001G00109000000,100.000,,MARKET ST,PITTSBURGH,PA,,15222.000,101,1st Ward - PITTSBURGH,47,Pittsburgh,LOT 39.99X60.19,PT EXTINGUISHED ALLEY = 4X40X4X40,NaN,51C02,PITTSBURGH URBAN,T,20 - Taxable,NaN,NaN,20,CORPORATION,C,COMMERCIAL,400,VACANT COMMERCIAL LAND,2568,NaN,NaN,NaN,NaN,09-03-2004,09-03-2004,344250.000,H,MULTI-PARCEL SA,12182,537.000,04-21-2004,67987.000,04-14-1997,1.000,PO BOX 669,,CARNEGIE PA,15106.000,0,65300,65300,0,0,65300,65300,0,65300,65300,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2025,2025-08-01
3,14151262,0001G00111000000,161.000,,1ST AVE,PITTSBURGH,PA,,15222.000,101,1st Ward - PITTSBURGH,47,Pittsburgh,GREGG PLAN OF LOTS,LOT 1 = 33.17X30X33.17X30,NaN,51C01,PITTSBURGH URBAN,T,20 - Taxable,NaN,NaN,10,REGULAR,R,RESIDENTIAL,10,SINGLE FAMILY,995,HOM,NaN,NaN,NaN,11-12-2021,11-10-2021,0.000,3,LOVE&AFFECTION,18681,572.000,10-02-2018,2000000.000,10-14-2015,1754901.000,161 1ST AVE,,PITTSBURGH PA,15222.000,1477100,119400,1596500,0,1495100,119400,1614500,1495100,119400,1614500,9.000,TOWNHOUSE,1.000,2015.000,2.000,Brick,4.000,ROLL,5.000,Full,X+,EXCELLENT +,1.000,EXCELLENT,EX,EXCELLENT,5.000,3.000,3.000,1.000,B,Central Heat with AC,NaN,NaN,2011.000,1.000,NaN,2025,2025-08-01
4,14151263,0001G00112000000,101.000,,MARKET ST,PITTSBURGH,PA,,15222.000,101,1st Ward - PITTSBURGH,47,Pittsburgh,GREGG PLAN OF LOTS,LOT 2 = 30X37.07X30X37.07,NaN,51C01,PITTSBURGH URBAN,T,20 - Taxable,NaN,NaN,10,REGULAR,R,RESIDENTIAL,10,SINGLE FAMILY,1112,NaN,NaN,NaN,NaN,10-15-2015,10-15-2015,1796575.000,9,OTHER INVALID,16161,462.000,01-06-2011,289800.000,01-06-1995,1.000,101 MARKET ST,,PITTSBURGH PA,15222.000,1398960,38300,1437260,0,1398960,38300,1437260,1398960,38300,1437260,9.000,TOWNHOUSE,3.000,2012.000,2.000,Br

In [10]:
appeals1 = appeals1[['PARID', 'SALEDATE', 'SALEPRICE', 'PREVSALEDATE', 'PREVSALEPRICE', 'PREVSALEDATE2', 'PREVSALEPRICE2']]
#appeals1[['PARID', 'ASOFDATE', 'FAIRMARKETTOTAL']]

appeals1.columns = ['parcelID', 'date0', 'value0', 'date1', 'value1', 'date2', 'value2'] # '0' most recent

In [11]:
temp = pd.DataFrame()
for i in range(3):
    slice = appeals1[['parcelID', f'date{i}', f'value{i}']]
    slice.columns = ['parcelID', 'date', 'value']
    temp = pd.concat([temp, slice], axis = 0)
appeals1 = temp.copy()

### Appeals 2

In [ ]:
# https://data.wprdc.org/dataset/property-data-with-geographic-identifiers/resource/8eff881d-4d28-4064-83f1-30cc991cfec7
# dictionary to store temporary dfs
dfs = {}

# download the data via the provided API
offset = 0
index = 0
while True:
    url = 'https://data.wprdc.org/api/3/action/datastore_search?resource_id=8eff881d-4d28-4064-83f1-30cc991cfec7&limit=32000'
    url += '&offset=' + str(offset)
    fileobj = urllib.request.urlopen(url)
    data = pd.DataFrame(json.loads(fileobj.read())['result']['records'])
    dfs['df' + str(index)] = data
    offset += 32000
    index += 1
    if len(data) == 0:
        break

appeals2 = pd.concat(dfs.values(), ignore_index = True)

In [ ]:
appeals2.shape

In [ ]:
appeals2.head(1)

,_id,PARID,PROPERTYHOUSENUM,PROPERTYFRACTION,PROPERTYADDRESS,PROPERTYCITY,PROPERTYSTATE,PROPERTYUNIT,PROPERTYZIP,MUNICODE,MUNIDESC,SCHOOLCODE,SCHOOLDESC,LEGAL1,LEGAL2,LEGAL3,NEIGHCODE,NEIGHDESC,TAXCODE,TAXDESC,TAXSUBCODE,TAXSUBCODE_DESC,OWNERCODE,OWNERDESC,CLASS,CLASSDESC,USECODE,USEDESC,LOTAREA,HOMESTEADFLAG,CLEANGREEN,FARMSTEADFLAG,ABATEMENTFLAG,RECORDDATE,SALEDATE,SALEPRICE,SALECODE,SALEDESC,DEEDBOOK,DEEDPAGE,PREVSALEDATE,PREVSALEPRICE,PREVSALEDATE2,PREVSALEPRICE2,CHANGENOTICEADDRESS1,CHANGENOTICEADDRESS2,CHANGENOTICEADDRESS3,CHANGENOTICEADDRESS4,COUNTYBUILDING,COUNTYLAND,COUNTYTOTAL,COUNTYEXEMPTBLDG,LOCALBUILDING,LOCALLAND,LOCALTOTAL,FAIRMARKETBUILDING,FAIRMARKETLAND,FAIRMARKETTOTAL,STYLE,STYLEDESC,STORIES,YEARBLT,EXTERIORFINISH,EXTFINISH_DESC,ROOF,ROOFDESC,BASEMENT,BASEMENTDESC,GRADE,GRADEDESC,CONDITION,CONDITIONDESC,CDU,CDUDESC,TOTALROOMS,BEDROOMS,FULLBATHS,HALFBATHS,HEATINGCOOLING,HEATINGCOOLINGDESC,FIREPLACES,BSMTGARAGE,FINISHEDLIVINGAREA,CARDNUMBER,ALT_ID,TAXYEAR,ASOFDATE,MUNICIPALITY,NEIGHBORHOOD,PGH_COUNCIL_DISTRICT,PGH_WARD,PGH_PUBLIC_WORKS_DIVISION,PGH_POLICE_ZONE,PGH_FIRE_ZONE,TRACT,BLOCK_GROUP
0,1,0001G00224060300,151.000,,FORT PITT BLVD,PITTSBURGH,PA,UNIT 603,15222,101,1st Ward - PITTSBURGH,47,Pittsburgh,FIRST SIDE CONDOMINIUM - AMENDMENT OF DECLARAT...,UNIT 603 & PARKING SPACE,6TH LEVEL,61P03H,151 FIRST,T,20 - Taxable,NaN,NaN,12,REGULAR-ETUX OR ET VIR,R,RESIDENTIAL,50,CONDOMINIUM,0,HOM,NaN,NaN,NaN,10-05-2017,09-29-2017,699000.000,36,QUIT CLAIM / SPEC WARRNTY,16966,186.000,10-25-2013,460000.000,08-23-2007,389745.000,151 FORT PITT BLVD UNIT 603,,PITTSBURGH PA,15222.000,592900,0,592900,0,610900,0,610900,610900,0,610900,21.000,CONDO HR,1.000,2007.000,7.000,Concrete,4.000,ROLL,1.000,NaN,A+,VERY GOOD +,3.000,AVERAGE,AV,AVERAGE,5.000,2.000,2.000,0.000,B,Central Heat with AC,NaN,NaN,1761.000,1.000,NaN,2018,2018-09-01,Pittsburgh,Central Business District,6.000,1.000,6.000,2.000,1-4,42003020100.000,420030201001.000


In [ ]:
appeals2.to_csv(raw + 'appeals2.csv', index = False)

In [7]:
appeals2 = pd.read_csv(raw + 'appeals2.csv')

In [8]:
appeals2

,_id,PARID,PROPERTYHOUSENUM,PROPERTYFRACTION,PROPERTYADDRESS,PROPERTYCITY,PROPERTYSTATE,PROPERTYUNIT,PROPERTYZIP,MUNICODE,MUNIDESC,SCHOOLCODE,SCHOOLDESC,LEGAL1,LEGAL2,LEGAL3,NEIGHCODE,NEIGHDESC,TAXCODE,TAXDESC,TAXSUBCODE,TAXSUBCODE_DESC,OWNERCODE,OWNERDESC,CLASS,CLASSDESC,USECODE,USEDESC,LOTAREA,HOMESTEADFLAG,CLEANGREEN,FARMSTEADFLAG,ABATEMENTFLAG,RECORDDATE,SALEDATE,SALEPRICE,SALECODE,SALEDESC,DEEDBOOK,DEEDPAGE,PREVSALEDATE,PREVSALEPRICE,PREVSALEDATE2,PREVSALEPRICE2,CHANGENOTICEADDRESS1,CHANGENOTICEADDRESS2,CHANGENOTICEADDRESS3,CHANGENOTICEADDRESS4,COUNTYBUILDING,COUNTYLAND,COUNTYTOTAL,COUNTYEXEMPTBLDG,LOCALBUILDING,LOCALLAND,LOCALTOTAL,FAIRMARKETBUILDING,FAIRMARKETLAND,FAIRMARKETTOTAL,STYLE,STYLEDESC,STORIES,YEARBLT,EXTERIORFINISH,EXTFINISH_DESC,ROOF,ROOFDESC,BASEMENT,BASEMENTDESC,GRADE,GRADEDESC,CONDITION,CONDITIONDESC,CDU,CDUDESC,TOTALROOMS,BEDROOMS,FULLBATHS,HALFBATHS,HEATINGCOOLING,HEATINGCOOLINGDESC,FIREPLACES,BSMTGARAGE,FINISHEDLIVINGAREA,CARDNUMBER,ALT_ID,TAXYEAR,ASOFDATE,MUNICIPALITY,NEIGHBORHOOD,PGH_COUNCIL_DISTRICT,PGH_WARD,PGH_PUBLIC_WORKS_DIVISION,PGH_POLICE_ZONE,PGH_FIRE_ZONE,TRACT,BLOCK_GROUP
0,1,0001G00224060300,151.000,,FORT PITT BLVD,PITTSBURGH,PA,UNIT 603,15222,101,1st Ward - PITTSBURGH,47,Pittsburgh,FIRST SIDE CONDOMINIUM - AMENDMENT OF DECLARAT...,UNIT 603 & PARKING SPACE,6TH LEVEL,61P03H,151 FIRST,T,20 - Taxable,NaN,NaN,12,REGULAR-ETUX OR ET VIR,R,RESIDENTIAL,50,CONDOMINIUM,0,HOM,NaN,NaN,NaN,10-05-2017,09-29-2017,699000.000,36,QUIT CLAIM / SPEC WARRNTY,16966,186.000,10-25-2013,460000.000,08-23-2007,389745.000,151 FORT PITT BLVD UNIT 603,,PITTSBURGH PA,15222.000,592900,0,592900,0,610900,0,610900,610900,0,610900,21.000,CONDO HR,1.000,2007.000,7.000,Concrete,4.000,ROLL,1.000,NaN,A+,VERY GOOD +,3.000,AVERAGE,AV,AVERAGE,5.000,2.000,2.000,0.000,B,Central Heat with AC,NaN,NaN,1761.000,1.000,NaN,2018,2018-09-01,Pittsburgh,Central Business District,6.000,1.000,6.000,2.000,1-4,42003020100.000,420030201001.000
1,2,0001G00224060400,151.000,,FORT PITT BLVD,PITTSBURGH,PA,UNIT 604,15222,101,1st Ward - PITTSBURGH,47,Pittsburgh,FIRST SIDE CONDOMINIUM - AMENDMENT OF DECLARAT...,UNIT 604 & PARKING SPACE,6TH LEVEL,61P03H,151 FIRST,T,20 - Taxable,NaN,NaN,11,REGULAR-ETAL,R,RESIDENTIAL,50,CONDOMINIUM,0,HOM,NaN,NaN,NaN,08-20-2012,08-20-2012,350000.000,AA,SALE NOT ANALYZ,14983,438.000,08-23-2007,350000.000,NaN,NaN,151 FORT PITT BLVD UNIT 604,,PITTSBURGH PA,15222.000,295600,0,295600,0,313600,0,313600,313600,0,313600,21.000,CONDO HR,1.000,2007.000,7.000,Concrete,4.000,ROLL,1.000,NaN,A+,VERY GOOD +,3.000,AVERAGE,AV,AVERAGE,4.000,2.000,2.000,0.000,B,Central Heat with AC,NaN,NaN,1275.000,1.000,NaN,2018,2018-09-01,Pittsburgh,Central Business District,6.000,1.000,6.000,2.000,1-4,42003020100.000,420030201001.000
2,3,0001G00224060500,151.000,,FORT PITT BLVD,PITTSBURGH,PA,UNIT 605,15222,101,1st Ward - PITTSBURGH,47,Pittsburgh,FIRST SIDE CONDOMINIUM - AMENDMENT OF DECLARAT...,UNIT 605 & PARKING SPACE,6TH LEVEL,61P03H,151 FIRST,T,20 - Taxable,NaN,NaN,12,REGULAR-ETUX OR ET VIR,R,RESIDENTIAL,50,CONDOMINIUM,0,NaN,NaN,NaN,NaN,09-18-2009,09-18-2009,265000.000,UR,OTHER VALID,14046,390.000,07-30-2007,235000.000,NaN,NaN,151 FORT PITT BLVD UNIT #605,,PITTSBURGH PA,15222.000,227800,0,227800,0,227800,0,227800,227800,0,227800,21.000,CONDO HR,1.000,2007.000,7.000,Concrete,4.000,ROLL,1.000,NaN,A+,VERY GOOD +,3.000,AVERAGE,AV,AVERAGE,3.000,1.000,1.000,1.000,B,Central Heat with AC,NaN,NaN,888.000,1.000,NaN,2018,2018-09-01,Pittsburgh,Central Business District,6.000,1.000,6.000,2.000,1-4,42003020100.000,420030201001.000
3,4,0002M00222000000,40.000,,VAN BRAAM ST,PITTSBURGH,PA,,15219,101,1st Ward - PITTSBURGH,47,Pittsburgh,LOT 25X54 VAN BRAAM ST BET FORBES & TUSTIN STS,2 STY BRK HSE,NaN,10101,BLUFF DISTRICT OF PITTSBURGH,T,20 - Taxable,NaN,NaN,12,REGULAR-ETUX OR ET VIR,R,RESIDENTIAL,10,SINGLE FAMILY,1350,NaN,NaN,NaN,NaN,NaN,08-14-1995,25550.000,4,OTHER,9518,404.000,NaN,NaN,NaN,NaN,203 EDELWEISS DR,,WEXFORD PA,15090.000,15700,8900,24600,0,15700,8900,24600,15700

In [13]:
appeals2 = appeals2[['PARID', 'SALEDATE', 'SALEPRICE', 'PREVSALEDATE', 'PREVSALEPRICE', 'PREVSALEDATE2', 'PREVSALEPRICE2']]
#appeals2[['PARID', 'ASOFDATE', 'FAIRMARKETTOTAL']]

appeals2.columns = ['parcelID', 'date0', 'value0', 'date1', 'value1', 'date2', 'value2'] # '0' most recent

In [14]:
temp = pd.DataFrame()
for i in range(3):
    slice = appeals2[['parcelID', f'date{i}', f'value{i}']]
    slice.columns = ['parcelID', 'date', 'value']
    temp = pd.concat([temp, slice], axis = 0)
appeals2 = temp.copy()

### Appeals 3

In [ ]:
# https://data.wprdc.org/dataset/allegheny-county-property-assessment-appeals/resource/8a7607fb-c93e-4d7a-9b23-528b5c25b1de
# dictionary to store temporary dfs
dfs = {}

# download the data via the provided API
offset = 0
index = 0
while True:
    url = 'https://data.wprdc.org/api/3/action/datastore_search?resource_id=8a7607fb-c93e-4d7a-9b23-528b5c25b1de&limit=32000'
    url += '&offset=' + str(offset)
    fileobj = urllib.request.urlopen(url)
    data = pd.DataFrame(json.loads(fileobj.read())['result']['records'])
    dfs['df' + str(index)] = data
    offset += 32000
    index += 1
    if len(data) == 0:
        break

appeals3 = pd.concat(dfs.values(), ignore_index = True)

In [ ]:
appeals3.shape

In [ ]:
appeals3.head(1)

In [ ]:
appeals3.to_csv(raw + 'appeals3.csv', index = False)

In [15]:
appeals3 = pd.read_csv(raw + 'appeals3.csv')

In [16]:
appeals3 = appeals3[['PARCEL ID', 'DISPO DATE', 'CURRENT TOTAL VALUE']]

appeals3.columns = ['parcelID', 'date', 'value']

### Concatenating values (all appeals + sales)

In [ ]:
sales.head(1)

In [ ]:
appeals1.head(1)

In [ ]:
appeals2.head(1)

In [ ]:
appeals3.head(1)

In [17]:
values = pd.concat([appeals1, appeals2, appeals3, sales], axis = 0)
values['date'] = pd.to_datetime(values['date'], format = 'mixed')

In [18]:
# merging in lat, lng - also filtering for Pittsburgh parcels
values = values.merge(right = parcels, on = 'parcelID', how = 'right')

# sort by date ascending
values = values.sort_values(by = 'date')

# filter dates including 2016 to be able to do smoothing for 2020
values = values.loc[(values['date'].dt.year >= 2016) & (values['date'].dt.year <= 2025)]

# removing abnormally large values (x > $10mm) to smooth results
values = values.loc[values['value'] <= 10_000_000]

# remove year-by-year duplicates
values = (
    values
    .assign(year = values['date'].dt.year)
    .drop_duplicates(subset = ['parcelID', 'year'], keep = 'last')
)

# drop nulls
values = values.dropna()

# reset the final index
values = values.reset_index(drop = True)

In [19]:
values.to_csv(clean + 'clean_values.csv', index = False)

In [17]:
values = pd.read_csv(clean + 'clean_values.csv')